In [ ]:
# ==============================================================================
# 1. CONFIGURACIÓN DE ENTORNO AUTOMÁTICA
# ==============================================================================
import os
import sys

try:
    # Detectar si estamos en Google Colab
    from google.colab import drive
    print("-> Entorno detectado: Google Colab.")
    
    # INSTALACIÓN DE MNE (Ya que no viene por defecto en Colab)
    print("-> Instalando librería MNE neuroscience...")
    !pip install mne -q
    
    # Montar Google Drive
    drive.mount('/content/drive')
    
    PATH_DRIVE = '/content/drive/MyDrive'
    REPO_NAME = 'bci-stroke-rehab'  # Asegúrate de que coincida con tu repo de GitHub
    PATH_REPO = os.path.join(PATH_DRIVE, REPO_NAME)
    URL_GITHUB = "https://github.com/tu-usuario/bci-stroke-rehab.git"
    
    if not os.path.exists(PATH_REPO):
        print(f"-> Clonando repositorio en: {PATH_REPO}...")
        os.chdir(PATH_DRIVE)
        !git clone {URL_GITHUB}
    else:
        print(f"-> El repositorio ya existe en tu Drive: {PATH_REPO}")
    
    os.chdir(PATH_REPO)
    sys.path.append(PATH_REPO)
    print(f"-> ¡Éxito! Directorio de trabajo configurado en: {os.getcwd()}")

except ImportError:
    print("-> Entorno detectado: Local PC. Asegúrate de haber activado tu bci_env.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
import os

# Importamos tu módulo personalizado de ingeniería de características
from src.eeg_datasets.preprocessed_dataset import PreprocessedDataset
from src.features_engineering.power_features import SignalProcessor

# Configuración estética de los gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 5]

In [ ]:
# ==============================================================================
# 2. SELECCIÓN DE BASE DE DATOS Y METADATOS
# ==============================================================================

# --- PARÁMETROS DE SELECCIÓN ---
DB_SELECCIONADA = "Cho2017"  # Opciones: "Cho2017" o "Dreyer2023A"
SUBJECTS = [50, 1, 3, 5 ] # Si ponemos None devuelve todos
CHANNELS = ["C3", "Cz", "C4"]
SESSIONS = ["session_1"]
DATA_TO_LOAD = ["motor_imagery"]
SFREQ = 128

# Instanciamos la basede Datos
dataset = PreprocessedDataset(db_name = DB_SELECCIONADA,
                              sessions = SESSIONS,
                              subjects = SUBJECTS,
                              data_to_load = DATA_TO_LOAD,
                              channels = CHANNELS,)

print(f"--- ANALIZANDO: Base de Datos [{DB_SELECCIONADA.upper()}] ---")

SUJETO_SELECCIONADO = 50
# --- CARGA REAL ---
X_eeg, Y_labels = dataset.flatten_pool_data(subject_ids = [SUJETO_SELECCIONADO]) # Si usan mas de un sujeto devolvera todos los datos de todos los sujetos

# --- IMPRESIÓN DE METADATOS (REQUISITO OBJETIVO) ---
print(f"✔ Sujeto actualmente seleccionado para análisis: {SUJETO_SELECCIONADO}")
print(f"✔ Cantidad de Canales críticos filtrados: {X_eeg.shape[1]}")
print(f"✔ Mapeo de Canales: {CHANNELS}")
print(f"✔ Dimensiones de la matriz EEG del sujeto [Ensayos, Canales, Tiempo]: {X_eeg.shape}")

In [ ]:
# ==============================================================================
# 3. VISUALIZACIÓN DE TRIALS (Mano Izquierda vs Mano Derecha)
# ==============================================================================

# Extraemos el primer ensayo disponible de la clase 0 (Izquierda) y clase 1 (Derecha)
trial_izq = X_eeg[Y_labels == 0][0]
trial_der = X_eeg[Y_labels == 1][0]
N_times = X_eeg.shape[-1] # La ultima dimension es la temporal. 

tiempo = np.arange(N_times) / SFREQ # Vector de tiempo en segundos

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f"Señales de EEG en el Dominio del Tiempo - Sujeto: {SUJETO_SELECCIONADO}", fontsize=15, fontweight='bold')

for idx, canal in enumerate(CHANNELS):
    axes[idx].plot(tiempo, trial_izq[idx], label="Imaginación Motora: Mano IZQUIERDA", color="crimson", alpha=0.8)
    axes[idx].plot(tiempo, trial_der[idx], label="Imaginación Motora: Mano DERECHA", color="teal", alpha=0.8)
    
    axes[idx].set_title(f"Canal {canal}", fontsize=12, fontweight='bold', loc='left')
    axes[idx].set_ylabel("Amplitud (µV)")
    axes[idx].legend(loc="upper right")
    axes[idx].grid(True, linestyle="--", alpha=0.5)

axes[2].set_xlabel("Tiempo (segundos)")
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 4. DENSIDAD ESPECTRAL DE POTENCIA (PSD) COMPARADA (C3 vs C4)
# ==============================================================================

# Calculamos la PSD promedio de TODOS los ensayos para el sujeto seleccionado
# Usamos el procesador de señales preprogramado en la carpeta src/
frecuencias, psd_completa = SignalProcessor.calcular_psd(X_eeg, sfreq=SFREQ)

# Mapeamos los índices de los canales de interés
idx_c3 = CHANNELS.index("C3")
idx_c4 = CHANNELS.index("C4")

# Promediamos los espectros a lo largo de todos los ensayos (eje 0) para graficar una tendencia limpia
psd_promedio_c3 = np.mean(psd_completa[:, idx_c3, :], axis=0)
psd_promedio_c4 = np.mean(psd_completa[:, idx_c4, :], axis=0)

# Graficamos ambos canales juntos en la misma figura para contrastarlos
plt.figure(figsize=(12, 5))
plt.plot(frecuencias, psd_promedio_c3, label="Canal C3 (Hemisferio Izquierdo)", color="darkblue", linewidth=2)
plt.plot(frecuencias, psd_promedio_c4, label="Canal C4 (Hemisferio Derecho)", color="darkorange", linewidth=2)

plt.title(f"Comparativa del Espectro de Frecuencias (PSD) - Canales C3 vs C4", fontsize=14, fontweight='bold')
plt.xlabel("Frecuencia (Hz)", fontsize=11)
plt.ylabel("Densidad de Potencia Espectral (µV² / Hz)", fontsize=11)

# Acotamos el gráfico a las bandas de interés neurofisiológico (Mu y Beta: 8 a 30 Hz) para guiar la vista
plt.xlim(4, 40) 
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# 5. EXTRACCIÓN DE CARACTERÍSTICAS (FEATURE ENGINEERING)
# ==============================================================================

print("-> Ejecutando extracción de Potencia Total Promedio por ensayo...")

# Invocamos la función matemática de la clase SignalProcessor
potencia_bruta = SignalProcessor.calcular_potencia_total(X_eeg)

print(f"\n[OK] ¡Pipeline de features verificado!")
print(f" -> Forma original de los datos RAW: {X_eeg.shape}")
print(f" -> Forma de la nueva matriz de Características [Ensayos, Características por canal]: {potencia_bruta.shape}")

# Creamos un DataFrame rápido para mostrarles cómo estructurar los datos para Machine Learning
df_features = pd.DataFrame(potencia_bruta, columns=[f"Potencia_{ch}" for ch in CHANNELS])
df_features['Target'] = y_mock
df_features['Label_Name'] = df_features['Target'].map({0: 'Mano_Izquierda', 1: 'Mano_Derecha'})

print("\nPrimeras filas del dataset listo para pasarle a un modelo de Scikit-Learn:")
display(df_features.head())